# Finetuning or Training Spotiflow on Custom Data

## Overview

[GitHub](https://github.com/weigertlab/spotiflow) | [Paper](https://www.nature.com/articles/s41592-026-02662-x) | [Spotiflow Documentation](https://weigertlab.org/spotiflow/index.html) | [Spotiflow API](https://weigertlab.org/spotiflow/api.html#)

In this notebook, we will walk through how to **finetune** a `Spotiflow` model following the instructions from the `Spotiflow` [example notebook](https://github.com/weigertlab/spotiflow/blob/main/examples/3_finetune.ipynb).

This process is very similar to training a model from scratch, but instead of initializing a new model with random weights, we will start with a pretrained model and finetune it on our data.

This is useful when the default models don't perform well on your data. Finetuning or training from scratch allows `Spotiflow` to learn directly from your examples, leading to better spot detection accuracy and more relevant results for your experiments.

You can find more details in the `Spotiflow` documentation for [training](https://weigertlab.org/spotiflow/train.html) and [finetuning](https://weigertlab.org/spotiflow/finetune.html).

## Make sure you have GPU access

To Enable GPU:

1 - Navigate to `Runtime -> Change Runtime Type`

2 - Select `Python 3` as `Runtime Type`

3 - Select one available GPU (e.g. `T4 GPU`) as `Hardware accelerator`.

<br>

<div align="left"> <img src="https://raw.githubusercontent.com/bobiac/bobiac-book/main/_static/images/cellpose/colab_runtime.png" alt="Ilastik Logo" width="400"></div>

## Mount your google drive

To access the data for the course you first need to mount your Google Drive.

Run the cell below to connect your Google Drive to colab and follow the instructions to authenticate your Google account.

You will need to allow access to your Google Drive so that the notebook can read and write files.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Then click on `folder icon` on the left bar, press the `refresh button`. Your Google Drive folder should now be available here (e.g. MyDrive).

<div align="left"> <img src="https://raw.githubusercontent.com/bobiac/bobiac-book/main/_static/images/cellpose/colab_folder.png" alt="Ilastik Logo" width="300"></div>

## Data

To go through the training process, you need **pairs of images and corresponding spot annotations**.

The **spot annotations** should be `.csv` files containing the spot coordinates organised in 2 column, `x` and `y` for 2D data and `z`, `x` and `y` for 3D data:

| x | y |
|---|---|
| 100.5 | 200.3 |
| 150.2 | 250.1 |

or, for 3D data:

| z | x | y |
|---|---|---|
| 10.0 | 100.5 | 200.3 |
| 15.0 | 150.2 | 250.1 |


The **image files** and their **corresponding spot annotation files MUST** have the same `name` and **MUST** be organized in the same folder. The data should be split into `train` and `val` folders, with an optional `test` folder:

```
spots_data
├── train
│   ├── img_001.csv
│   ├── img_001.tif
│   ...
│   ├── img_002.csv
│   └── img_002.tif
├── val
│   ├── val_img_001.csv
│   ├── val_img_001.tif
│   ...
│   ├── val_img_002.csv
│   └── val_img_002.tif
└── test (optional)
    ├── test_img_001.csv
    ├── test_img_001.tif
    ...
    ├── test_img_002.csv
    └── test_img_002.tif
```

For this notebook, we will use the `Spotiflow Finetuning Dataset` suggested in the `Spotiflow` [example notebook](https://github.com/weigertlab/spotiflow/blob/main/examples/3_finetune.ipynb) (MERFISH dataset from [Zhang et al, 2021](https://doi.org/10.1038/s41586-021-03705-x)).

## Download the Data

Run the cell below to download the data for this exercise and save it in you Google Drive. A new folder called `spotiflow_finetuning_dataset` will be created in your Google Drive.

In [ ]:
# Create directory
!mkdir -p /content/spotiflow_finetuning_dataset
# Download the data
!wget https://github.com/bobiac/bobiac-book/releases/download/data-bobiac-2026/05_spot_detection_spotiflow_finetuning.zip -O /content/spotiflow_finetuning_dataset/05_spot_detection_spotiflow_finetuning.zip
# Unzip the data, remove zip file and macOS metadata files (if any)
!cd /content/spotiflow_finetuning_dataset && unzip 05_spot_detection_spotiflow_finetuning.zip && rm -f 05_spot_detection_spotiflow_finetuning.zip && rm -rf __MACOSX

## Install Dependencies

In [ ]:
!pip install spotiflow
!pip install matplotlib
!pip install tqdm

## Import Libraries

## Load the Data

Since the data we will use are organized as described above, to load the data ans split them in training, validation and test sets, we can simply use the [`get_data`](https://weigertlab.org/spotiflow/api.html#spotiflow.utils.get_data) function provided by the `Spotiflow` API, which will automatically look for the `.tif` and `.csv` files in the specified folders and create the appropriate data structures for training.

Specify the `include_test` argument to `True` if you have a `test` folder with data that you want to use for testing after training.

We can then visualize an example image and its corresponding spot annotations to verify that the data has been loaded correctly.

`train_imgs` and `train_spots` (and the corresponding `val` and `test` variables) are **lists** of images and spot coordinates, therefore we can index them to visualize one example.

## Finetune a Pretrained Model

We can now finetune a pretrained `Spotiflow` model on our data.

### Load a Pretrained Model

The first step is to load the pretrained model using the [`Spotiflow.from_pretrained()`](https://weigertlab.org/spotiflow/api.html#spotiflow.model.spotiflow.Spotiflow.from_pretrained) method, which allows you to specify the name of the pretrained model you want to load.

For this example, we will finetune the `synth_complex` model (automatically downloaded if not already present on your system).

<p class="alert alert alert-info">
    <strong>Note:</strong> If you want to load a model from a path, you can instead use the <a href="https://weigertlab.org/spotiflow/api.html#spotiflow.model.spotiflow.Spotiflow.from_folder" target="_blank"><code>Spotiflow.from_folder()</code></a> method and specify the path to the folder where the model is located.
</p>

<p class="alert alert alert-info">
    <strong>Note:</strong> If you want to train a model from scratch instead of finetuning a pretrained model, you can first define a new model configuration using the <a href="https://weigertlab.org/spotiflow/api.html#spotiflow.model.config.SpotiflowModelConfig" target="_blank"><code>SpotiflowModelConfig</code></a> class and then initialize a new model using the <a href="https://weigertlab.org/spotiflow/api.html#spotiflow.model.spotiflow.SpotiflowTrainingConfig"><code>SpotiflowTrainingConfig</code></a> class. See the <a href="https://github.com/weigertlab/spotiflow/blob/main/examples/1_train.ipynb" target="_blank">Spotiflow Training notebook</a> for more details.
</p>


### Prepare the Training Configuration

The next step is to prepare the training configuration using the [`SpotiflowTrainingConfig`](https://weigertlab.org/spotiflow/api.html#spotiflow.model.config.SpotiflowTrainingConfig) class.

This step is identical to the one used for training a model from scratch.

Here we will only change a few parameters, but you can find the full description of the training configuration in the dropdown below.

<details>
<summary><b>Spotiflow <code>SpotiflowTrainingConfig</code> Parameters</b></summary>
<br>

**Optimization**

| Parameter | Type | Default | What it does |
|:---|:---|:---|:---|
| `lr` | `float` | `3e-4` | Learning rate for the AdamW optimizer. Controls how large each weight update is. When **finetuning**, consider lowering it (e.g. `1e-4`) so you don't erase what the pretrained model already knows; when training **from scratch**, the default is usually a good starting point. |
| `optimizer` | `str` | `"adamw"` | Optimization algorithm. Currently only `"adamw"` is supported. |
| `batch_size` | `int` | `4` | Number of crops processed simultaneously before each weight update. Larger batches give more stable gradients but use more GPU memory; reduce it if you run out of memory. |
| `num_epochs` | `int` | `200` | Number of full passes over the (sampled) training data. More epochs give the model more time to learn, but too many can lead to *overfitting*. Watch the validation loss: if it starts rising while the training loss keeps falling, you've trained too long. For a quick tutorial run, a much smaller value (e.g. `30`) is enough. |
| `lr_reduce_patience` | `int` | `10` | Number of epochs without validation-loss improvement before the learning rate is automatically reduced (`ReduceLROnPlateau`). Set to `0` to disable the reduction. |
| `early_stopping_patience` | `int` | `0` | Number of epochs without validation-loss improvement before training stops early. `0` disables early stopping (train for the full `num_epochs`). |

**Sampling & Cropping**

| Parameter | Type | Default | What it does |
|:---|:---|:---|:---|
| `crop_size` | `int \| tuple[int, ...]` | `512` | Side length (in pixels) of the random square crops extracted from each image during training. Internally clamped to fit your images and the network's minimum size, so for small images the effective crop may be smaller. |
| `crop_size_depth` | `int` | `32` | Crop size along the Z axis, used only for **3D** data. Ignored for 2D. |
| `smart_crop` | `bool \| float` | `True` | If `True`, crops are preferentially sampled around annotated spots (so crops are less likely to be empty); `False` samples crops uniformly at random. You can also pass a float in `[0, 1]` to set the sampling bias directly. Useful when spots are sparse. |
| `num_train_samples` | `int \| None` | `None` | Number of crops sampled per epoch. `None` uses one crop per training image. Setting it higher than your dataset size (e.g. `200` for a handful of images) keeps the number of updates per epoch constant and lets each image be sampled multiple times with different augmentations, which helps on small datasets. |

**Loss**

| Parameter | Type | Default | What it does |
|:---|:---|:---|:---|
| `heatmap_loss_f` | `str` | `"bce"` | Loss function for the spot heatmap. One of `"bce"`, `"mse"`, `"smoothl1"`, or `"adawing"`. |
| `flow_loss_f` | `str` | `"l1"` | Loss function for the stereographic flow regression. Currently only `"l1"` is supported. |
| `pos_weight` | `float` | `10.0` | Weight given to positive (spot) pixels in the heatmap loss. Spots are sparse, so positive pixels are up-weighted to counteract the class imbalance. Increase it if the model misses faint spots, decrease it if it over-detects. |
| `loss_levels` | `int \| None` | `None` | Number of resolution levels at which the heatmap loss is computed. `None` uses all the model's levels. Must be ≤ the model's number of levels. |

**Other**

| Parameter | Type | Default | What it does |
|:---|:---|:---|:---|
| `finetuned_from` | `str \| None` | `None` | Metadata string recording which pretrained model this one was finetuned from. Stored in the saved config; does not affect training itself. |
| `optimize_threshold` | `bool` | `False` | If `True`, automatically tunes the spot-detection probability threshold on the validation set at the end of training, so prediction works well out of the box without manual threshold tuning. |

</details>

### Finetune the Model
We can now train the model with calling the model's [`fit()`](https://weigertlab.org/spotiflow/api.html#spotiflow.model.spotiflow.Spotiflow.fit) method.

We can specify different parameters such as the *training* and *validation* data, the training **configuration** we defined in the previous step, and the **path** where to save the finetuned model.

### Evaluate the Finetuned Model

Now that the model is finetuned, we can evaluate its performance on the test set when compared to the pretrained model.

We first need to load the `synth_complex` model and then run both this and the finetuned models on the test set to compare their performance.

Let's now visualize the predictions of both models to visually compare their performance.